# 4.18 Random projections

Random projections turns unlabeled data into structure by choosing the right notion of similarity, compression, or surprise.

Part 4 moves from prediction with labels to discovery without labels. Vectors, covariance, and matrix factorization become the language for finding hidden low-dimensional structure. Random projections keep the geometry story but remove PCA's training step: a seeded random map is evaluated by whether neighborhoods survive.

Save a copy to Drive to edit.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.decomposition import NMF
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.manifold import MDS
from sklearn.manifold import SpectralEmbedding
from sklearn.manifold import TSNE
from sklearn.manifold import trustworthiness
from sklearn.metrics import pairwise_distances
from sklearn.random_projection import GaussianRandomProjection
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

def dimred_ladder():
    """D1..D5 dimensionality-reduction ladder. Returns [(name, X, y), ...] of rising ambient dim.

    2-D toy -> 3-D swiss-roll-ish -> digits(64-D) -> the same with noise dims -> a wide feature set.
    y is a color/label for visualization only.
    """
    rungs = []
    rng = np.random.default_rng(3)

    t = np.linspace(0, 4, 120)
    x1 = np.column_stack([t, 0.5 * t + rng.normal(0, 0.05, 120)])
    rungs.append(("D1 near-1-D line in 2-D", x1, t))

    tt = np.linspace(0, 3 * np.pi, 200)
    x2 = np.column_stack([tt * np.cos(tt), 8 * rng.random(200), tt * np.sin(tt)])
    rungs.append(("D2 swiss-roll (3-D)", x2, tt))

    digits = load_digits()
    rungs.append(("D3 digits (real, 64-D)", digits.data / 16.0, digits.target))

    xn = np.hstack([digits.data / 16.0, rng.normal(0, 1, size=(digits.data.shape[0], 32))])
    rungs.append(("D4 digits + 32 noise dims", xn, digits.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (30-D)", bc.data, bc.target))

    return rungs



def sample_for_embedding(X, y, max_points=500, seed=SEED):
    rng = np.random.default_rng(seed)
    if X.shape[0] <= max_points:
        return X, y
    idx = rng.choice(X.shape[0], size=max_points, replace=False)
    order = np.sort(idx)
    return X[order], y[order]


def standardize_for_geometry(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X)


def nonnegative_for_factorization(X):
    scaler = MinMaxScaler()
    return scaler.fit_transform(X)


def safe_trustworthiness(X, Z):
    neighbors = min(10, max(1, (X.shape[0] - 1) // 3))
    return float(trustworthiness(X, Z, n_neighbors=neighbors))


def plot_ladder_embeddings(results, metric_name):
    fig, axes = plt.subplots(1, len(results), figsize=(17, 3.4))
    for ax, item in zip(axes, results):
        scatter = ax.scatter(item["Z"][:, 0], item["Z"][:, 1], c=item["y"], s=10, cmap="viridis")
        ax.set_title(item["name"].split("(")[0], fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle("2D embeddings across the D1-D5 ladder")
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 3.5))
    xs = np.arange(1, len(results) + 1)
    ys = [item["metric"] for item in results]
    ax.plot(xs, ys, marker="o")
    ax.set_xticks(xs)
    ax.set_xticklabels([f"D{i}" for i in xs])
    ax.set_ylabel(metric_name)
    ax.set_xlabel("dataset rung")
    ax.set_title(f"{metric_name} vs. ladder complexity")
    ax.grid(True, alpha=0.3)
    plt.show()


def preview_ladder(rungs):
    for i, (name, X, y) in enumerate(rungs, start=1):
        unique = np.unique(y)
        label_info = len(unique) if unique.size < 30 else "continuous"
        print(f"D{i}: {name} | X={X.shape} | color/label info={label_info}")
        print(np.round(X[:3, :min(5, X.shape[1])], 3))

## The concept, built once: PCA baseline arithmetic before a random map

The lesson's audit formula is $$X_c=U\Sigma V^	op,\qquad Z=X_cV_r$$. Random projection does not choose $V_r$ by SVD, but the SVD baseline gives a concrete variance number to compare against.

In [ ]:
toy = np.array([[1.0, 2.0], [2.0, 1.0], [3.0, 4.0], [4.0, 3.0]])
means = toy.mean(axis=0)
centered = toy - means
_, singular_values, vt = np.linalg.svd(centered, full_matrices=False)
energies = singular_values ** 2
first_energy = energies[0] / energies.sum()
print("column means", means)
print("singular values", singular_values)
print("first-direction energy", first_energy)
assert np.allclose(means, [2.5, 2.5])
assert np.allclose(np.round(singular_values, 3), [2.828, 1.414])
assert np.allclose(np.round(energies, 3), [8.0, 2.0])
assert np.isclose(first_energy, 0.8)

Now build the reusable method. A Gaussian random projection samples a seeded matrix $R$ and returns $Z=XR$; the useful question is whether distances and neighborhoods survive, not whether variance is optimal.

In [ ]:
def method(X, n_components=2, seed=SEED):
    X_scaled = standardize_for_geometry(X)
    n_components = min(n_components, X_scaled.shape[1])
    projector = GaussianRandomProjection(n_components=n_components, random_state=seed)
    Z = projector.fit_transform(X_scaled)
    if Z.shape[1] == 1:
        Z = np.column_stack([Z[:, 0], np.zeros(Z.shape[0])])
    return Z

Z_toy = method(toy, n_components=1, seed=0)
print("toy projection shape", Z_toy.shape)
assert centered.shape == (4, 2)
assert Z_toy.shape == (4, 2)

## The dataset ladder

We use the shared F3 dimensionality-reduction ladder: a near-1D toy line, a 3D swiss-roll-style surface, real handwritten digits, digits with added noise dimensions, and a real 30-dimensional breast-cancer feature table. The labels are only colors for visualization; the embedding method does not train on them.

In [ ]:
rungs = dimred_ladder()
preview_ladder(rungs)

## Run the same method across D1-D5

Each rung is standardized or scaled in the same way, embedded into 2D, and scored with trustworthiness. Subsampling is seeded and only bounds future notebook runtime; this build script does not execute the notebook.

In [ ]:
metric_name = "trustworthiness"
results = []
for rung_index, (name, X, y) in enumerate(rungs, start=1):
    X_small, y_small = sample_for_embedding(X, y, max_points=500, seed=SEED + rung_index)
    Z = method(X_small, n_components=2, seed=SEED + rung_index)
    score = safe_trustworthiness(standardize_for_geometry(X_small), Z)
    results.append({"name": name, "X": X_small, "y": y_small, "Z": Z, "metric": score})
    print(f"D{rung_index} | {name:32s} | trustworthiness={score:.3f}")

## Results visualization

The closing figure has two parts: small-multiple embedding panels for D1-D5, then a metric curve as the data become more realistic and higher dimensional.

In [ ]:
plot_ladder_embeddings(results, metric_name)

## Pitfall on the hardest rung: random seed variability

The lesson warns that stability checks matter. On D5, two seeded random maps can tell slightly different visual stories, so the fix is to average several seeds and inspect distance preservation instead of trusting one panel.

In [ ]:
name, X, y = rungs[-1]
X_small, y_small = sample_for_embedding(X, y, max_points=500, seed=9)
Z_a = method(X_small, seed=1)
Z_b = method(X_small, seed=2)
wrong_a = safe_trustworthiness(standardize_for_geometry(X_small), Z_a)
wrong_b = safe_trustworthiness(standardize_for_geometry(X_small), Z_b)
seed_scores = []
for seed in range(10):
    Z_seed = method(X_small, seed=seed)
    seed_scores.append(safe_trustworthiness(standardize_for_geometry(X_small), Z_seed))
print("single-seed scores", round(wrong_a, 3), round(wrong_b, 3))
print("multi-seed mean", round(float(np.mean(seed_scores)), 3))
print("multi-seed std", round(float(np.std(seed_scores)), 3))

## Evaluate it + Practice

- Report trustworthiness next to a no-skill baseline such as plotting two raw standardized features or a random 2D map.
- Sanity check that nearby points in the original space still have nearby points in the embedding.
- Ablate the key idea: remove scaling, change the random seed, or use too few neighbors/components and watch the metric move.
- Watch for failure signals: unstable layouts, one feature dominating distances, disconnected neighbor graphs, or a better objective with worse held-out structure.
- Treat labels as a post-hoc audit only; unsupervised methods do not get to train on them.


### Practice

Compare Gaussian random projection with the first two raw standardized features on D5.

In [ ]:
# Your code here


### Practice

Increase the target dimension to 5 before plotting a PCA view of the projected data; what happens to trustworthiness?

In [ ]:
# Your code here


### Practice

Repeat the seed sweep with and without StandardScaler and explain the difference.

In [ ]:
# Your code here
